In [17]:
!python --version

Python 3.11.13


재구성

In [18]:
!pip uninstall -y tf-keras keras-nightly keras==3.* tensorflow==2.16.* tensorflow==2.17.* tensorflow==2.18.* tf-nightly


Found existing installation: keras 2.15.0
Uninstalling keras-2.15.0:
  Successfully uninstalled keras-2.15.0
Found existing installation: tensorflow 2.15.0.post1
Uninstalling tensorflow-2.15.0.post1:
  Successfully uninstalled tensorflow-2.15.0.post1


In [19]:
!pip -q install "numpy==1.26.4" "ml-dtypes==0.2.0" "h5py==3.10.0"


In [20]:
%env TF_USE_LEGACY_KERAS=1
!pip -q install "tensorflow==2.15.0.post1" "keras==2.15.0"


env: TF_USE_LEGACY_KERAS=1


In [21]:
!pip -q install --no-deps "deepctr==0.9.3"


재시작

In [1]:

import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# 내부 경로에 Keras 2 심볼 매핑
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# init_ops_v2 대체 (DeepCTR가 참조)
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

print("TF:", tf.__version__, "| Keras:", keras.__version__, "→ shim ready")

TF: 2.15.0 | Keras: 2.15.0 → shim ready


In [2]:
import tensorflow as tf, keras, deepctr
print("TF:", tf.__version__)
print("Keras:", keras.__version__)
print("DeepCTR:", deepctr.__version__)

from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
from deepctr.models import DeepFM

print("DeepCTR import OK")


TF: 2.15.0
Keras: 2.15.0
DeepCTR: 0.9.3
DeepCTR import OK


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')
test = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/test.parquet' , engine= 'pyarrow')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


test 열에 없는 열 train에서 버리기

In [5]:
target = 'clicked'

cols_to_keep = [c for c in train.columns if c in test.columns or c == target]

train = train[cols_to_keep]

train , test 데이터 타입 맞추기

- test 데이터 id열 제외하고 float32로




In [6]:
cols_to_convert = [c for c in test.columns if c not in ['seq' ,'ID']]
test[cols_to_convert]  = test[cols_to_convert].astype("float32")

In [7]:
train.info(verbose = True , show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3203885 entries, 0 to 3203884
Data columns (total 119 columns):
 #    Column        Non-Null Count    Dtype  
---   ------        --------------    -----  
 0    gender        3203885 non-null  float32
 1    age_group     3203885 non-null  float32
 2    inventory_id  3203885 non-null  float32
 3    day_of_week   3203885 non-null  float32
 4    hour          3203885 non-null  float32
 5    seq           3203885 non-null  object 
 6    l_feat_1      3203885 non-null  float32
 7    l_feat_2      3203885 non-null  float32
 8    l_feat_3      3203885 non-null  float32
 9    l_feat_4      3203885 non-null  float32
 10   l_feat_5      3203885 non-null  float32
 11   l_feat_6      3203885 non-null  float32
 12   l_feat_7      3203885 non-null  float32
 13   l_feat_8      3203885 non-null  float32
 14   l_feat_9      3203885 non-null  float32
 15   l_feat_10     3203885 non-null  float32
 16   l_feat_11     3203885 non-null  float32
 17   l_feat

In [8]:
test.info(verbose = True , show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1527298 entries, 0 to 1527297
Data columns (total 119 columns):
 #    Column        Non-Null Count    Dtype  
---   ------        --------------    -----  
 0    ID            1527298 non-null  object 
 1    gender        1526362 non-null  float32
 2    age_group     1526362 non-null  float32
 3    inventory_id  1527298 non-null  float32
 4    day_of_week   1527298 non-null  float32
 5    hour          1527298 non-null  float32
 6    seq           1527298 non-null  object 
 7    l_feat_1      1527298 non-null  float32
 8    l_feat_2      1526362 non-null  float32
 9    l_feat_3      1527298 non-null  float32
 10   l_feat_4      1527298 non-null  float32
 11   l_feat_5      1527298 non-null  float32
 12   l_feat_6      1527298 non-null  float32
 13   l_feat_7      1527298 non-null  float32
 14   l_feat_8      1526362 non-null  float32
 15   l_feat_9      1527298 non-null  float32
 16   l_feat_10     1527298 non-null  float32
 17   l_feat

메타데이터 저장

In [9]:
train["is_train"] = 1
test["is_train"]  = 0

test_id = test["ID"].copy()

/tmp/ipython-input-9-1840936110.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["is_train"]  = 0


In [10]:
all_data = pd.concat([train, test], ignore_index=True)

In [11]:
df = all_data.copy()

In [12]:
df.info(verbose = True , show_counts= True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4731183 entries, 0 to 4731182
Data columns (total 121 columns):
 #    Column        Non-Null Count    Dtype  
---   ------        --------------    -----  
 0    gender        4730247 non-null  float32
 1    age_group     4730247 non-null  float32
 2    inventory_id  4731183 non-null  float32
 3    day_of_week   4731183 non-null  float32
 4    hour          4731183 non-null  float32
 5    seq           4731183 non-null  object 
 6    l_feat_1      4731183 non-null  float32
 7    l_feat_2      4730247 non-null  float32
 8    l_feat_3      4731183 non-null  float32
 9    l_feat_4      4731183 non-null  float32
 10   l_feat_5      4731183 non-null  float32
 11   l_feat_6      4731183 non-null  float32
 12   l_feat_7      4731183 non-null  float32
 13   l_feat_8      4730247 non-null  float32
 14   l_feat_9      4731183 non-null  float32
 15   l_feat_10     4731183 non-null  float32
 16   l_feat_11     4731183 non-null  float32
 17   l_feat

In [13]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

    max_len : 800
    topk : 1000
    min_count = 500

In [27]:
import numpy as np
import pandas as pd

MAX_LEN = 150
PAD_ID = 0  # 0은 PAD, 실제 토큰은 +1 오프셋

def build_seq_padded_len(series: pd.Series, max_len: int, *, offset: int = 1, ignore_neg: bool = True):
    """
    series: 콤마 구분 문자열("9,18,269,...") 컬럼
    offset=1: 모델 전처리처럼 +1 오프셋(0은 PAD로 예약)
    return: seq_padded(int32, [N,max_len]), seq_len(int32, [N]), vocab_size(int)
    """
    # 미리 결과 배열을 한 번에 할당(메모리/속도 핵심)
    N = len(series)
    seq_padded = np.zeros((N, max_len), dtype=np.int32)
    seq_len    = np.zeros(N, dtype=np.int32)
    max_id     = 0

    # 판다스 오버헤드 줄이기: 바로 넘파이 배열로
    # astype(str)을 쓰면 NaN -> 'nan' 문자열이 되므로, 아래에서 비어 있으면 건너뜀
    vals = series.to_numpy(copy=False)

    for i in range(N):
        s = vals[i]
        if s is None or (isinstance(s, float) and np.isnan(s)):
            # 빈 시퀀스
            continue

        # 문자열로 캐스팅 (np.fromstring은 공백을 무시하므로 replace 불필요)
        text = s if isinstance(s, str) else str(s)
        if not text:
            continue

        # C 가속 파싱: 매우 빠름. 실패하면 size=0
        arr = np.fromstring(text, sep=',', dtype=np.int64)
        if arr.size == 0:
            continue

        if ignore_neg:
            # 음수 제거(있다면)
            arr = arr[arr >= 0]
            if arr.size == 0:
                continue

        if offset:
            # +1 오프셋 (0=PAD 유지)
            arr = arr + offset

        # 트렁케이팅: 최신 항목을 남기고 앞을 자름 (pre-truncating)
        L = arr.size
        if L > max_len:
            arr = arr[-max_len:]
            L = max_len

        # 패딩된 행에 앞쪽부터 복사 (post-padding)
        # arr는 int64이므로 복사 시 자동 캐스팅 → 비용 적음
        seq_padded[i, :L] = arr
        seq_len[i] = L

        # vocab_size 계산용 최대 id 갱신 (한 번에 끝)
        amax = int(arr.max()) if L > 0 else 0
        if amax > max_id:
            max_id = amax

    vocab_size = int(max_id + 1)  # PAD 포함
    return seq_padded, seq_len, vocab_size

# 사용 예시
seq_padded, seq_len, vocab_size = build_seq_padded_len(df["seq"], MAX_LEN, offset=1, ignore_neg=True)

# # df에 길이만 저장
# df["seq_len"] = seq_len


In [28]:
PAD_ID   = 0
MAX_LEN  = 150
OFFSET   = 1


tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)


# 1) train만으로 vocab “fit”
seq_tr_pad, seq_tr_len, vocab_seq = build_seq_padded_len(
    df.loc[tr_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg=True
)

# 2) test는 같은 규칙으로 transform만 + OOV 클램핑
seq_te_pad, seq_te_len, _ = build_seq_padded_len(
    df.loc[te_mask, "seq"], MAX_LEN, offset=OFFSET, ignore_neg=True
)

# train에서 결정한 vocab_seq를 기준으로, 범위 밖 토큰(>= vocab_seq)은 0으로
seq_tr_pad = np.where(seq_tr_pad < vocab_seq, seq_tr_pad, PAD_ID).astype("int32")
seq_te_pad = np.where(seq_te_pad < vocab_seq, seq_te_pad, PAD_ID).astype("int32")

# 길이 저장(원하면 df에도 반영)
df.loc[tr_mask, "seq_len"] = seq_tr_len
df.loc[te_mask, "seq_len"] = seq_te_len


### 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.


<br>


순서형의 경우 임베딩은 순서 정보를 기억하지 않는다.
>
    조회만 한다: e_k = Embedding[k] — k는 의미 있는 수가 아니라 “키”.

    순열 불변성: 첫 레이어가 ŷ = W·e_k + b일 때, 라벨을 임의로 섞고(순열 P) 임베딩과 가중치를 같이 섞으면 ŷ가 그대로.
    즉 모델은 라벨 순서에 무관하게 동치 해를 가짐.

    그래디언트 독립: 각 카테고리 벡터가 독립적으로 업데이트되어 연속성/단조성이 보장되지 않음. “2는 1과 3 사이”라는 규칙을 스스로 학습하리란 보장이 없다.

    그래서 임베딩으로 처리하면 ‘명목형처럼’ 취급되고, 순서(ordinal) 정보는 구조적으로 전달되지 않는다.

> 두 개의 표현을 동시에 사용 - SparseFeat(범주형) 과 DenseFeat(순서형 숫자) 둘 다 넣기.



<BR>

hour , day_of_week 열에 관하여

>

    day_of_week: 순서 중요, 월~일 주기성 있음

    hour: 시간대도 순서형이라 DenseFeat 가능. 다만 주기성(23시→0시)이 있어서 사인/코사인 변환.




In [29]:
label_feat = ['gender', 'age_group', 'day_of_week' , 'inventory_id' , 'hour']

encoders = {}

for feat in label_feat:
    le = LabelEncoder()
    df[feat] = le.fit_transform(df[feat])
    encoders[feat] = le

< 계획 >

1. 결측치 보강


2. **hour, day_of_week**

 >
    사인/코사인 변환 + 이중표현(Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착)

    Sparse: 원래 카테고리도 함께 넣어 요일/시간대별 개별 패턴 포착.
>
    장점: 주기성(23→0의 연속성) + 카테고리별 임베딩 패턴을 동시에 잡는다.



3. 연속형 추정 열들

>

    l_feat_3, l_feat_27, feat_e_4, feat_a_1, feat_a_3, feat_a_4, feat_a_8, feat_a_13, feat_a_16, feat_a_18
>
    Dense: 결측치 imputer(median 등) → MinMaxScaler 후 그대로 입력.

    Sparse(해시): 결측치 보강 후 -> 바로  처리


> 이중화의 경우 과적합의 위험이 있으니 embedding_dim을 작게(4~8), dnn_dropout, l2_reg_embedding 등을 적절히 사용

In [30]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, KBinsDiscretizer

####### 1번
ord_cols  = ["gender","hour", "day_of_week", "age_group", "inventory_id" ,'l_feat_14']

cont_cols = ["l_feat_3","l_feat_27","feat_e_4",
             "feat_a_1","feat_a_3","feat_a_4","feat_a_8",
             "feat_a_13","feat_a_16","feat_a_18"]


seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

ban = set(ord_cols + cont_cols + [seq_col, label_col, seq_len_col, "is_train", "ID"])

# 진또베기 연속형들
extra_cont_cols = [c for c in df if c not in ban]

# 순서형 + 연속형
cont_all_cols = cont_cols + extra_cont_cols

# dense용 전체 (ord + cont + 찐연속형)
num_all_cols = ord_cols + cont_all_cols


# train / test 분리
tr_mask = df["is_train"].eq(1)
te_mask = df["is_train"].eq(0)


# train으로만 fit : test 결측치 대체할 통계 학습
imp_ord = SimpleImputer(strategy="most_frequent").fit(df.loc[tr_mask , ord_cols])
imp_cont = SimpleImputer(strategy="median").fit(df.loc[tr_mask, cont_all_cols])

Xord_tr  = imp_ord.transform(df.loc[tr_mask,ord_cols])
Xcont_tr = imp_cont.transform(df.loc[tr_mask,cont_all_cols])

Xnum_tr = np.hstack([Xord_tr , Xcont_tr])


# Dense용 스케일러 - train 단계에서 fit만해서 따로 저장
scaler = MinMaxScaler().fit(Xnum_tr)




# train에서만 mp 생성
nuniq    = {c: np.unique(Xcont_tr[:, cont_all_cols.index(c)]).size for c in cont_cols}
map_cols = [c for c, u in nuniq.items() if u <= 15]  # 임계값 10~15 권장

mp_cont = {
    c: {v: i+1 for i, v in enumerate(np.unique(Xcont_tr[:, cont_all_cols.index(c)]))}
    for c in map_cols
}

In [31]:
############# 2번 transform 함수 (train/test 공용)

def make_dense_sparse(
    _df,
    mask,
    *,
    ord_cols,               # ["hour","day_of_week","age_group","inventory_id"]
    cont_all_cols,          # cont_cols + extra_cont_cols
    num_all_cols,           # ord_cols + cont_all_cols
    imp_ord, imp_cont,      # 각각 most_frequent, median
    scaler,                 # MinMaxScaler (train으로만 fit)
    map_cols, mp_cont       # 저카디널 연속형만 정수 매핑
):


    Xord  = imp_ord.transform(_df.loc[mask, ord_cols])
    Xcont = imp_cont.transform(_df.loc[mask, cont_all_cols])
    Xnum  = np.hstack([Xord, Xcont])
    Xnum_s = scaler.transform(Xnum)

    #  Dense: 스케일된 값 + 주기형 sin/cos(스케일 전 값으로)
    dense = pd.DataFrame(Xnum_s, columns = num_all_cols , index=_df.index[mask])

    # 스케일 전 값으로 cos/sin 계산
    hour_vals = Xnum[:, ord_cols.index("hour")]
    dow_vals  = Xnum[:, ord_cols.index("day_of_week")]

    dense["hour_sin"] = np.sin(2*np.pi*hour_vals/24)
    dense["hour_cos"] = np.cos(2*np.pi*hour_vals/24)

    dense["dow_sin"]  = np.sin(2*np.pi*dow_vals/7)
    dense["dow_cos"]  = np.cos(2*np.pi*dow_vals/7)


    dense_cols = num_all_cols + ["hour_sin","hour_cos","dow_sin","dow_cos"]



    # sprase : 보조적으로 sparse에 넣을 것들 정의
    sparse = pd.DataFrame(index=_df.index[mask])

    if 'gender' in _df.columns:
      g = _df.loc[mask, 'gender'].astype('float32')
      sparse['gender'] = g.where(g.isin([1,2]), np.nan).fillna(0).astype('int32') # 1,2 이외의 값은 결측치 0으로

    sparse["hour_cat"] = (_df.loc[mask, "hour"].astype("float64").fillna(-1).astype("int32") + 1) # -1→0(UNK), 0→1, 23→24
    sparse["day_of_week_cat"] = (_df.loc[mask, "day_of_week"].astype("float64").fillna(0).clip(lower=0, upper=7).astype("int32"))
    sparse["age_group_cat"] = (_df.loc[mask, "age_group"].astype("float64").fillna(-1).astype("int32") + 1)
    sparse["inventory_id_cat"]= _df.loc[mask, "inventory_id"].astype("Int64").fillna(0).astype("int32")


    # sparse_cols = ["hour_cat","day_of_week_cat","age_group_cat","gender","inventory_id_cat"]


    # 저카디널 연속형만 정수 매핑(임퓨트된 값 기준)
    for c in map_cols:
        j = cont_all_cols.index(c)
        v = pd.Series(Xcont[:, j], index=_df.index[mask])          # impute된 연속값
        sparse[c + "_cat"] = v.map(mp_cont[c]).fillna(0).astype("int32")


    base_sparse = ["hour_cat","day_of_week_cat","age_group_cat","inventory_id_cat"]

    if "gender" in sparse.columns:
        base_sparse = ["gender"] + base_sparse

    sparse_cols = base_sparse + [c + "_cat" for c in map_cols]

    return dense[dense_cols], sparse[sparse_cols], dense_cols, sparse_cols




ban = set(ord_cols + cont_cols + [seq_col, label_col, seq_len_col, "is_train", "ID", "gender"])
extra_cont_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in ban]

cont_all_cols = cont_cols + extra_cont_cols
num_all_cols  = ord_cols + cont_all_cols

dense_tr, sparse_tr, dense_cols, sparse_cols = make_dense_sparse(
    df, tr_mask,
    ord_cols=ord_cols, cont_all_cols=cont_all_cols, num_all_cols=num_all_cols,
    imp_ord=imp_ord, imp_cont=imp_cont, scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont
)
dense_te, sparse_te, _, _ = make_dense_sparse(
    df, te_mask,
    ord_cols=ord_cols, cont_all_cols=cont_all_cols, num_all_cols=num_all_cols,
    imp_ord=imp_ord, imp_cont=imp_cont, scaler=scaler,
    map_cols=map_cols, mp_cont=mp_cont
)

In [32]:
HASH_BUCKET = 200_000
MAX_LEN = 150

EMBED_DIM = 8 # 임시로 통일

def _vocab_from_int(s):
    # s: pd.Series(int). 반드시 0 이상.
    return int(s.max()) + 1 if len(s) else 1


sparse_fixed = [
    SparseFeat('gender',            vocabulary_size=_vocab_from_int(sparse_tr['gender']),            embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('inventory_id_cat',  vocabulary_size=_vocab_from_int(sparse_tr['inventory_id_cat']),  embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('hour_cat',          vocabulary_size=_vocab_from_int(sparse_tr['hour_cat']),          embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('day_of_week_cat',   vocabulary_size=_vocab_from_int(sparse_tr['day_of_week_cat']),   embedding_dim=EMBED_DIM, dtype='int32'),
    SparseFeat('age_group_cat',     vocabulary_size=_vocab_from_int(sparse_tr['age_group_cat']),     embedding_dim=EMBED_DIM, dtype='int32'),
]

sparse_hash = [
    SparseFeat('l_feat_14', vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , use_hash= True ,dtype='string'),
]


dense_feats = [DenseFeat(c, 1) for c in dense_cols]


# 순서형 - sparse에 넣기
for c in map_cols:
    name = f"{c}_cat"
    if name in sparse_tr.columns:
        sparse_fixed.append(
            SparseFeat(name, vocabulary_size=_vocab_from_int(sparse_tr[name]), embedding_dim=EMBED_DIM, dtype='int32')
        )


varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size = vocab_seq , # train 기준으로 vocab 계산(0=패딩/UNK 전제)
                            embedding_dim   = EMBED_DIM ,
                            # use_hash=True ,
                            dtype= 'int32', # 정수형 사용 -> 해시 안됨
                              ),
    maxlen   = MAX_LEN,
    combiner = 'mean',
    length_name = 'seq_len',
    weight_name = None,
    weight_norm = False
)




In [ ]:
fixlen_feature_columns = sparse_fixed + sparse_hash + dense_feats

if varlen_seq is not None:
    linear_feature_columns = fixlen_feature_columns + [varlen_seq]
    dnn_feature_columns    = fixlen_feature_columns + [varlen_seq]
else:
    linear_feature_columns = fixlen_feature_columns
    dnn_feature_columns    = fixlen_feature_columns

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용

# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭


In [34]:
# 검증용
assert seq_tr_pad.max() < vocab_seq and seq_te_pad.max() < vocab_seq

## 학습 샘플 생성 및 모델 학습

DeepCTR 모델은 내부적으로 특성 이름별로 Input Layer를 자동 생성한다.

그래서 입력을 dict 형태로 요구한다.


<br>
주의
>

    DataFrame의 seq는 건들지 말고, 모델에 넣을 때만 seq_padded/seq_len을 넘긴다


In [ ]:
# target = 'clicked'

# train_proc = df[df["is_train"] == 1].drop(columns=["is_train", "ID"])
# test_proc = df[df["is_train"] == 0].drop(columns=["is_train", "ID"])


# train_model_input = {name: train[name] for name in feature_names}
# test_model_input = {name: test[name] for name in feature_names}

# train_y = train[target].to_numpy()

In [ ]:
def to_inputs_fast(dense_df, sparse_df, dense_cols, sparse_cols, *, seq_pad=None, seq_len=None):
    X = {}
    # Sparse: 정수 인덱스 vs 문자열 해시 구분
    for c in sparse_cols:
        s = sparse_df[c]
        if str(s.dtype).startswith("string") or s.dtype == object:
            # DeepCTR 해시용: str 배열
            X[c] = s.astype("string").to_numpy(dtype=object, copy=False)
        else:
            # 정수 인덱스
            X[c] = s.to_numpy(dtype=np.int32, copy=False)
    # Dense: float32
    for c in dense_cols:
        X[c] = dense_df[c].to_numpy(dtype=np.float32, copy=False)
    # VarLen(있으면)
    if seq_pad is not None:
        X["seq"]     = np.asarray(seq_pad, dtype=np.int32)
    if seq_len is not None:
        X["seq_len"] = np.asarray(seq_len, dtype=np.int32)
    return X




X_train = to_inputs_fast(dense_tr, sparse_tr, dense_cols, sparse_cols,
                         seq_pad=seq_tr_pad if 'seq' in feature_names else None,
                         seq_len=seq_tr_len if 'seq' in feature_names else None)
X_test  = to_inputs_fast(dense_te, sparse_te, dense_cols, sparse_cols,
                         seq_pad=seq_te_pad if 'seq' in feature_names else None,
                         seq_len=seq_te_len if 'seq' in feature_names else None)

train_model_input = X_train
test_model_input  = X_test
train_y = df.loc[tr_mask, "clicked"].astype("float32").to_numpy()


In [ ]:
target = 'clicked'


train_seq_padded, train_seq_len, _ = build_seq_padded_len(train["seq"], MAX_LEN, offset=1, ignore_neg=True)
test_seq_padded, test_seq_len, _ = build_seq_padded_len(test["seq"], MAX_LEN, offset=1, ignore_neg=True)


train_model_input = {}
test_model_input = {}

for name in feature_names:
    if name == 'seq':
        train_model_input[name] = seq_tr_pad.astype('int32')
        test_model_input[name]  = seq_te_pad.astype('int32')

    elif name == 'seq_len':
        train_model_input[name] = seq_tr_len.astype('int32')
        test_model_input[name]  = seq_te_len.astype('int32')

    elif name in [f.name for f in sparse_hash]:
        train_model_input[name] = train[name].values.astype(object)
        test_model_input[name] = test[name].values.astype(object)

    elif name in [f.name for f in sparse_fixed]:
        train_model_input[name] = train[name].astype('int32').values
        test_model_input[name] = test[name].astype('int32').values

    else:  # DenseFeat
        train_model_input[name] = train[name].astype('float32').values
        test_model_input[name] = test[name].astype('float32').values


train_y = train[target].to_numpy()
# test_y = test[target].to_numpy()

In [ ]:
model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model.compile("adam", "binary_crossentropy",
              metrics=['binary_crossentropy'], )

In [ ]:
# ============================================
# DeepCTR on Colab (Py 3.11) — TF 2.15 / Keras 2.15 호환 셋업
# ============================================
# 사용 시점:
# 1) 노트북 맨 위 셀에서 실행
# 2) DeepCTR, 모델 정의/fit 전에 반드시 실행
# --------------------------------------------
# 필요 시 설치(이미 설치되었다면 주석 유지하세요)
# !pip install -q "tensorflow==2.15.0" "keras==2.15.0" "deepctr==0.9.3"

import os, sys, types

# 레거시 tf.keras 사용 (Keras 3 경로로 빠지지 않도록)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# ====== 출력(버전 확인) ======
print(f"TF: {tf.__version__} | Keras: {keras.__version__}")

# --------------------------------------------
# 1) DeepCTR가 기대하는 내부 경로 심볼 매핑 (tf.keras 내부 경로 → 외부 keras 심볼 프록시)
#    - tensorflow.python.keras.layers / initializers 를 keras.* 로 연결
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# 2) DeepCTR가 참조하는 init_ops_v2 대체 모듈 주입
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

# 3) TF 2.15에 없는 내부 심볼 주입:
#    model.fit 경로에서 isinstance(ds, input_lib.DistributedDatasetInterface) 체크 시 AttributeError 방지
from tensorflow.python.distribute import input_lib as _input_lib
if not hasattr(_input_lib, "DistributedDatasetInterface"):
    class DistributedDatasetInterface:
        """Minimal shim for TF 2.15; acts as a marker interface only."""
        pass
    _input_lib.DistributedDatasetInterface = DistributedDatasetInterface

print("→ Internal shims installed OK")

# --------------------------------------------
# 4) DeepCTR import 및 버전 확인
try:
    import deepctr
    from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
    from deepctr.models import DeepFM
    print(f"DeepCTR: {deepctr.__version__} (import OK)")
except Exception as e:
    print("DeepCTR import failed:", repr(e))
    raise

# (선택) 버전 가드 — 환경이 바뀌었을 때 빨리 눈치채기 위함
def _assert_versions(tf_req="2.15.", keras_req="2.15.", deepctr_req="0.9."):
    assert tf.__version__.startswith(tf_req), f"Require TF {tf_req}x, got {tf.__version__}"
    assert keras.__version__.startswith(keras_req), f"Require Keras {keras_req}x, got {keras.__version__}"
    assert deepctr.__version__.startswith(deepctr_req), f"Require DeepCTR {deepctr_req}x, got {deepctr.__version__}"

try:
    _assert_versions()
    print("→ Version guard passed (TF/Keras/DeepCTR expected range).")
except AssertionError as ae:
    print("! Version guard warning:", ae)

print("Shim setup complete. You can now build & fit DeepCTR models safely.")


In [ ]:
history = model.fit(train_model_input, train[target].values,
                    batch_size=256, epochs=10, verbose=2, validation_split=0.2, )
pred_ans = model.predict(test_model_input, batch_size=256)


print("test LogLoss", round(log_loss(test[target].values, pred_ans), 4))
print("test AUC", round(roc_auc_score(test[target].values, pred_ans), 4))

롤백 or 훈련 종료 후 평가?

In [ ]:
from sklearn.metrics import average_precision_score, log_loss
from sklearn.utils.class_weight import compute_class_weight

y_true = test[target].values.ravel().astype(np.int32)
y_pred = pred_ans.ravel().astype(float)

ap_pos = average_precision_score(y_true, y_pred)
ap_neg = average_precision_score(1 - y_true, 1 - y_pred)
ap_50  = 0.5 * (ap_pos + ap_neg)

w = compute_class_weight('balanced', classes=np.array([0,1]), y=y_true)
sw = np.where(y_true==0, w[0], w[1])
wll_50 = log_loss(y_true, y_pred, sample_weight=sw)

print(f"AP (50%): {ap_50:.4f}")
print(f"WLL(50%): {wll_50:.4f}")


AP (50%): 0.5296
WLL(50%): 1.8578


모델 저장

In [ ]:
save_dir = "/content/drive/MyDrive/Colab Notebooks/CRT/model"
import os
os.makedirs(save_dir, exist_ok=True)

# 저장: 체크포인트 형식(.index, .data-00000-of-00001 파일 세트)
ckpt_path = f"{save_dir}/best.weights"
model.save_weights(ckpt_path)          # 또는 model.save_weights(ckpt_path, save_format='tf')


In [ ]:
# # 로드: 동일한 모델 구조를 코드로 재생성한 뒤
model2 = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model2.compile(optimizer='adam', loss='binary_crossentropy')
model2.load_weights(ckpt_path)

제출 파일 만들기

In [ ]:
import pandas as pd
submission = pd.DataFrame({
    "ID": test_id,         # 보관해둔 test id
    "prediction": pred
})

submission.to_csv("submission.csv", index=False)
